# 02 — Tiền xử lý dữ liệu

Loại bản sao trước khi chia train/test 80/20. Chỉ chuẩn hóa Time và Amount bằng thống kê từ train; giữ nguyên dữ liệu gốc.

## 1. Thư viện và đường dẫn

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

## 2. Đọc và kiểm tra dữ liệu

In [3]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Kích thước dữ liệu gốc: {df.shape}")
display(df.head())

Kích thước dữ liệu gốc: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
target_column = "Class"
feature_columns = df.columns.drop(target_column).tolist()
expected_features = ["Time", *[f"V{i}" for i in range(1, 29)], "Amount"]

assert not df.empty, "Dataset rỗng."
assert len(feature_columns) == 30 and set(feature_columns) == set(expected_features)
assert df.select_dtypes(exclude="number").empty, "Có cột không phải kiểu số."
assert df.notna().all().all(), "Có giá trị thiếu; cần kiểm tra trước khi chia dữ liệu."
assert np.isfinite(df.to_numpy()).all(), "Có giá trị vô hạn."
assert set(df[target_column].unique()) == {0, 1}, "Nhãn phải gồm Class 0 và Class 1."

print("Biến mục tiêu:", target_column)
print(f"{len(feature_columns)} đặc trưng:", feature_columns)

Biến mục tiêu: Class
30 đặc trưng: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


## 3. Kiểm tra trùng lặp

Dòng trùng được xét trên toàn bộ 31 cột. Chỉ bỏ các bản sao sau lần xuất hiện đầu; không loại ngoại lệ theo Amount hoặc các đặc trưng V.

In [5]:
duplicate_mask = df.duplicated()
duplicate_counts = df.loc[duplicate_mask, target_column].value_counts().reindex([0, 1], fill_value=0)
class_counts_before = df[target_column].value_counts().reindex([0, 1], fill_value=0)

duplicate_summary = pd.DataFrame({
    "Trước xử lý": class_counts_before,
    "Bản sao loại bỏ": duplicate_counts,
    "Sau xử lý": class_counts_before - duplicate_counts,
})
duplicate_summary.index.name = "Class"
print(f"Số bản sao trùng hoàn toàn: {duplicate_mask.sum():,}")
display(duplicate_summary)

Số bản sao trùng hoàn toàn: 1,081


,Trước xử lý,Bản sao loại bỏ,Sau xử lý
Class,,,
0,284315,1062,283253
1,492,19,473


In [6]:
df_clean = df.drop_duplicates(keep="first").copy()
assert not df_clean.duplicated().any()
assert len(df) - len(df_clean) == duplicate_mask.sum()
print(f"Kích thước sau xử lý trùng lặp: {df_clean.shape}")

Kích thước sau xử lý trùng lặp: (283726, 31)


## 4. Tách X, y và chia train/test

In [7]:
X = df_clean.drop(columns=target_column)
y = df_clean[target_column]

assert not X.duplicated().any(), "Có đặc trưng giống nhau nhưng nhãn khác; cần kiểm tra trước khi chia."
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)

In [8]:
split_shapes = pd.DataFrame({
    "Kích thước": [X_train_raw.shape, X_test_raw.shape, y_train.shape, y_test.shape],
}, index=["X_train", "X_test", "y_train", "y_test"])
display(split_shapes)

,Kích thước
X_train,"(226980, 30)"
X_test,"(56746, 30)"
y_train,"(226980,)"
y_test,"(56746,)"


In [9]:
label_counts = pd.DataFrame({
    "Toàn bộ sau bỏ trùng": y.value_counts(),
    "Train": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).reindex([0, 1], fill_value=0)
label_counts.index.name = "Class"
label_ratios = label_counts.div(label_counts.sum(), axis="columns") * 100

display(label_counts)
display(label_ratios.round(4).rename(columns=lambda column: f"{column} (%)"))

,Toàn bộ sau bỏ trùng,Train,Test
Class,,,
0,283253,226602,56651
1,473,378,95


,Toàn bộ sau bỏ trùng (%),Train (%),Test (%)
Class,,,
0,99.8333,99.8335,99.8326
1,0.1667,0.1665,0.1674


### Kiểm tra giao nhau giữa train và test

In [10]:
assert X_train_raw.index.intersection(X_test_raw.index).empty
assert X_train_raw.index.equals(y_train.index)
assert X_test_raw.index.equals(y_test.index)
assert len(X_train_raw) + len(X_test_raw) == len(df_clean)
assert (label_counts["Train"] + label_counts["Test"]).equals(label_counts["Toàn bộ sau bỏ trùng"])

shared_rows = X_train_raw.merge(X_test_raw, on=feature_columns, how="inner")
assert shared_rows.empty, "Có giao dịch cùng đặc trưng xuất hiện ở cả train và test."
print(f"Số dòng có cùng đặc trưng ở cả hai tập: {len(shared_rows)}")

Số dòng có cùng đặc trưng ở cả hai tập: 0


## 5. Chuẩn hóa Time và Amount

Fit StandardScaler trên train, chỉ transform test. Giữ nguyên V1–V28, tên cột và thứ tự dòng/cột.

In [11]:
scale_columns = ["Time", "Amount"]
scaler = StandardScaler()
scaler.fit(X_train_raw[scale_columns])

scaler_statistics = pd.DataFrame({
    "Mean từ train": scaler.mean_,
    "Standard deviation từ train": scaler.scale_,
}, index=scale_columns)
display(scaler_statistics)

,Mean từ train,Standard deviation từ train
Time,94900.224390,47488.109673
Amount,88.387124,245.766505


In [12]:
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
X_train[scale_columns] = scaler.transform(X_train_raw[scale_columns])
X_test[scale_columns] = scaler.transform(X_test_raw[scale_columns])

display(X_train.head())

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
226238,1.045499,2.238954,-1.724499,-2.151484,-2.577803,0.993668,3.565492,-1.785957,0.860122,-1.264003,...,-0.323810,-0.149574,-0.049333,0.278442,0.684735,-0.219028,-0.159167,0.037920,-0.049932,-0.229434
134253,-0.298690,-1.315062,1.630783,0.597001,-0.038359,-0.404580,-0.965712,0.212249,0.735381,-1.267926,...,-0.067580,-0.238898,-0.946773,0.323904,0.515632,-0.713000,-0.266503,-0.017794,0.051058,-0.331197
186465,0.678397,1.908801,0.021184,-2.087997,0.129310,1.161468,0.605244,-0.022371,0.180296,0.283819,...,-0.210474,0.293609,1.095842,-0.044874,-1.689517,0.106098,0.007758,0.045164,-0.053068,-0.298809
149493,-0.074929,1.811257,0.316556,0.316751,3.880231,0.048454,1.020163,-0.734868,0.233651,0.681423,...,-0.228032,0.138869,0.700422,0.174064,0.702997,-0.212523,-0.010018,-0.017740,-0.038006,-0.289247
18461,-1.376728,1.358817,-1.120881,0.550266,-1.547659,-1.194950,0.275448,-1.201843,0.212889,-2.094285,...,-0.361686,-0.340972,-0.636442,0.252758,-0.344160,-0.064282,-0.439622,0.062524,0.013095,-0.261985


### Kiểm tra sau chuẩn hóa

In [13]:
v_columns = [f"V{i}" for i in range(1, 29)]
for processed, raw, labels in [(X_train, X_train_raw, y_train), (X_test, X_test_raw, y_test)]:
    assert processed.columns.tolist() == feature_columns
    assert processed.shape[1] == 30
    assert processed.index.equals(labels.index)
    assert processed.notna().all().all() and labels.notna().all()
    assert np.isfinite(processed.to_numpy()).all()
    pd.testing.assert_frame_equal(processed[v_columns], raw[v_columns])

assert scaler.n_samples_seen_ == len(X_train_raw)
assert scaler.feature_names_in_.tolist() == scale_columns
np.testing.assert_allclose(scaler.mean_, X_train_raw[scale_columns].mean(), rtol=1e-12)
np.testing.assert_allclose(X_train[scale_columns].mean(), 0, atol=1e-12)
np.testing.assert_allclose(X_train[scale_columns].std(ddof=0), 1, atol=1e-12)
print("Đạt: 30 cột, không NaN/vô hạn, V1–V28 giữ nguyên và scaler chỉ fit trên train.")

Đạt: 30 cột, không NaN/vô hạn, V1–V28 giữ nguyên và scaler chỉ fit trên train.


In [14]:
scaling_check = pd.DataFrame({
    "Train mean": X_train[scale_columns].mean(),
    "Train std (ddof=0)": X_train[scale_columns].std(ddof=0),
    "Test mean": X_test[scale_columns].mean(),
    "Test std (ddof=0)": X_test[scale_columns].std(ddof=0),
})
display(scaling_check.round(6))

,Train mean,Train std (ddof=0),Test mean,Test std (ddof=0)
Time,-0.0,1.0,-0.009386,0.999212
Amount,0.0,1.0,0.001741,1.090992


## 6. Lưu dữ liệu và scaler

CSV không lưu index; các file X và y tương ứng giữ cùng thứ tự dòng. X_train/X_test đã chuẩn hóa dùng chung cho ba mô hình. Các file có hậu tố `_raw` dùng khi cần cross-validation với Pipeline; scaler phải được fit lại trong từng fold. Chưa áp dụng SMOTE hay thay đổi phân phối test.

In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

output_tables = {
    "X_train.csv": X_train,
    "X_test.csv": X_test,
    "y_train.csv": y_train.to_frame(),
    "y_test.csv": y_test.to_frame(),
    "X_train_raw.csv": X_train_raw,
    "X_test_raw.csv": X_test_raw,
}
for filename, table in output_tables.items():
    table.to_csv(PROCESSED_DIR / filename, index=False)

SCALER_PATH = MODELS_DIR / "standard_scaler.joblib"
joblib.dump(scaler, SCALER_PATH)

['D:\\Ky1_Nam3\\ML\\CK\\credit-card-fraud-detection\\models\\standard_scaler.joblib']

## 7. Đọc lại và kiểm tra file đầu ra

In [16]:
for filename, expected in output_tables.items():
    saved_path = PROCESSED_DIR / filename
    assert saved_path.is_file() and saved_path.stat().st_size > 0
    loaded = pd.read_csv(saved_path, float_precision="round_trip")
    pd.testing.assert_frame_equal(loaded, expected.reset_index(drop=True), check_exact=True)

loaded_scaler = joblib.load(SCALER_PATH)
assert loaded_scaler.feature_names_in_.tolist() == scale_columns
assert loaded_scaler.n_samples_seen_ == len(X_train_raw)
np.testing.assert_array_equal(
    loaded_scaler.transform(X_test_raw[scale_columns]),
    X_test[scale_columns].to_numpy(),
)
print("Đạt: đọc lại đủ 6 CSV và scaler; giá trị, tên cột, nhãn và thứ tự dòng khớp hoàn toàn.")

Đạt: đọc lại đủ 6 CSV và scaler; giá trị, tên cột, nhãn và thứ tự dòng khớp hoàn toàn.


In [17]:
output_paths = [PROCESSED_DIR / filename for filename in output_tables] + [SCALER_PATH]
output_summary = pd.DataFrame({
    "File": [str(path.relative_to(PROJECT_ROOT)) for path in output_paths],
    "Dung lượng (MB)": [path.stat().st_size / 1024**2 for path in output_paths],
})
display(output_summary)
display(split_shapes)
display(label_counts[["Train", "Test"]])

,File,Dung lượng (MB)
0,data\processed\X_train.csv,120.002085
1,data\processed\X_test.csv,30.001929
2,data\processed\y_train.csv,0.649402
3,data\processed\y_test.csv,0.162358
4,data\processed\X_train_raw.csv,114.413705
5,data\processed\X_test_raw.csv,28.603835
6,models\standard_scaler.joblib,0.000892


,Kích thước
X_train,"(226980, 30)"
X_test,"(56746, 30)"
y_train,"(226980,)"
y_test,"(56746,)"


,Train,Test
Class,,
0,226602,56651
1,378,95


### Sử dụng ở giai đoạn 3

Đọc X và y tương ứng từ `data/processed/`; cột nhãn là `Class`. Dùng `_raw` với Pipeline nếu cross-validation, giữ test độc lập và không chuẩn hóa lần hai các file X đã xử lý. SMOTE (nếu dùng) chỉ áp dụng lên phần train của mỗi fold; chọn siêu tham số/ngưỡng trên validation, không trên test.